# Colab — Variants As and Bs refit

Trains the regime LSTMs and computes the ensembles for **two new pipelines**:

- **Variant As** = variant A's features (price/volume + AAII sentiment, 13-dim) but with a plain `GaussianHMM` instead of the BIC-optimal `GMMHMM(n_mix=2)` used by the original variant A.
- **Variant Bs** = variant B's features (variant A + VIX-family, 16-dim) but with a plain `GaussianHMM` instead of the BIC-optimal `GMMHMM(n_mix=2)` used by the original variant B.

**Why this exists.** The variant-Os refit (notebook `colab_variant_Os_refit.ipynb`) demonstrated that switching from GMMHMM(mix=2) to GaussianHMM dramatically improves the ensemble's absolute accuracy on variant O's features (MAPE 30.1% → 23.5%, the lowest of all 7 predictors), without making the regime-vs-baseline DM significant. To rule out the alternative explanation that A and B's existing GMMHMM partitions are themselves degraded, we train parallel GaussianHMM versions of A and B (`As`, `Bs`) and re-evaluate. Original variants A and B remain unchanged for direct comparison.

**Self-contained:** does NOT modify nb05 or nb06. Uses `src/train_LSTM_regime.py` directly and computes ensembles + DM tests inline. All output filenames use `_As` / `_Bs` suffixes so existing artifacts are never overwritten.

**Wall time:** ~50–70 min on Colab GPU (12 LSTM trainings: 2 variants × 3 seeds × 2 regimes; seed 42 runs full 20-trial Optuna per regime, seeds 43/44 reuse seed-42 hparams).

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DRIVE = '/content/drive/MyDrive/StockVolatilitySight'
os.makedirs(PROJECT_DRIVE, exist_ok=True)
print('Drive root:', PROJECT_DRIVE)

## Cell 2 — Clone repo (PAT entered securely via getpass)

In [ ]:
import getpass, subprocess, shutil, os

GH_USER   = 'maharajhaider'
GH_REPO   = 'StockVolatilitySight'
GH_BRANCH = 'iteration-and-report-analysis'
REPO_DIR  = '/content/repo'

if os.path.exists(REPO_DIR):
    print(f'{REPO_DIR} already exists — removing and re-cloning for a clean state.')
    shutil.rmtree(REPO_DIR)

github_pat = getpass.getpass('GitHub classic PAT (ghp_...): ').strip()
if not github_pat:
    raise RuntimeError('No PAT provided — clone would fail on private repo.')

clone_url = f'https://{github_pat}@github.com/{GH_USER}/{GH_REPO}.git'
result = subprocess.run(
    ['git', 'clone', '--branch', GH_BRANCH, '--single-branch', clone_url, REPO_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    err = result.stderr.replace(github_pat, '<PAT>')
    raise RuntimeError(f'git clone failed:\n{err}')

os.chdir(REPO_DIR)
print(f'Cloned to {REPO_DIR}')
print(subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
del github_pat, clone_url

## Cell 3 — Install requirements

In [ ]:
!pip install -q -r requirements.txt

## Cell 4 — Restore data + variant A and B baseline LSTMs from Drive

Required from Drive:
1. `data/processed/{train,val,test}.parquet`, `features_all.parquet`, `raw_dataset.parquet`
2. Variant A baseline LSTMs `models/lstm_baseline_seed{42,43,44}.pt` + scalers
3. Variant B baseline LSTMs `models/lstm_baseline_B_seed{42,43,44}.pt` + scalers

(The baseline LSTMs are HMM-independent so we reuse the existing variant A and B baselines for As and Bs respectively — same features, no need to retrain.)

Required in cloned repo:
- `models/hmm_*_As.joblib`, `models/hmm_*_Bs.joblib`
- `data/processed/regime_probabilities_As.parquet`, `regime_probabilities_Bs.parquet`

In [ ]:
import shutil
from pathlib import Path

src_proc   = Path(PROJECT_DRIVE) / 'data' / 'processed'
src_models = Path(PROJECT_DRIVE) / 'models'
dst_proc   = Path(REPO_DIR) / 'data' / 'processed'
dst_models = Path(REPO_DIR) / 'models'
(dst_proc / 'seeds').mkdir(parents=True, exist_ok=True)
dst_models.mkdir(parents=True, exist_ok=True)

REQUIRED_DATA = ['train.parquet', 'val.parquet', 'test.parquet',
                 'features_all.parquet', 'raw_dataset.parquet']
for fname in REQUIRED_DATA:
    src = src_proc / fname
    if src.exists():
        shutil.copy2(src, dst_proc / fname)
        print(f'  data: {fname}')
    else:
        print(f'  data: {fname}  (MISSING in Drive)')

# variant A baseline (no suffix in legacy naming) and variant B baseline (_B suffix)
for suffix in ('', '_B'):
    for seed in (42, 43, 44):
        for stem in (f'lstm_baseline{suffix}_seed{seed}.pt',
                     f'lstm_baseline{suffix}_seed{seed}_scaler.joblib'):
            src = src_models / stem
            if src.exists():
                shutil.copy2(src, dst_models / stem)
                print(f'  baseline: {stem}')
            else:
                print(f'  baseline: {stem}  (MISSING — needed for ensemble)')

required_in_repo = [
    dst_models / 'hmm_winner_As.joblib', dst_models / 'hmm_meta_As.joblib', dst_models / 'hmm_scaler_As.joblib',
    dst_models / 'hmm_winner_Bs.joblib', dst_models / 'hmm_meta_Bs.joblib', dst_models / 'hmm_scaler_Bs.joblib',
    dst_proc   / 'regime_probabilities_As.parquet',
    dst_proc   / 'regime_probabilities_Bs.parquet',
]
missing = [f for f in required_in_repo if not f.exists()]
if missing:
    raise FileNotFoundError(
        'Missing variant As / Bs HMM artifacts in the cloned repo:\n  ' +
        '\n  '.join(str(m) for m in missing) +
        '\nRun the local refit script and push the new files to the branch first.')
print('\nAll required variant As / Bs HMM artifacts present.')

## Cell 5 — Sanity check: both HMMs are GaussianHMM with clean regimes

In [ ]:
import joblib, pandas as pd
for tag in ('As', 'Bs'):
    m    = joblib.load(f'models/hmm_winner_{tag}.joblib')
    meta = joblib.load(f'models/hmm_meta_{tag}.joblib')
    rp   = pd.read_parquet(f'data/processed/regime_probabilities_{tag}.parquet')
    autocorr = rp['p_volatile'].autocorr(1)
    print(f'Variant {tag}: class={type(m).__name__}, p_stay={m.transmat_.diagonal().round(3)}, '
          f'autocorr lag-1={autocorr:.3f}')
    assert type(m).__name__ == 'GaussianHMM', f'{tag}: expected GaussianHMM, got {type(m).__name__}'
    assert autocorr > 0.85, f'{tag}: autocorr {autocorr:.3f} too low — wrong artifacts loaded'
print('\n✓ Both As and Bs HMMs are GaussianHMM with clean regime persistence.')

## Cell 6 — Train both variants' regime LSTMs (3 seeds × 2 regimes per variant)

12 LSTM trainings total. Each variant's seed 42 runs the full Optuna search (~10 min/variant), seeds 43 and 44 reuse seed-42 hparams (~3 min each).

LSTM input features:
- **Variant As** uses `LSTM_VARIANT_A_FEATURES` (7 features = 5 stationary + bullish + bearish)
- **Variant Bs** uses `LSTM_VARIANT_B_FEATURES` (10 features = variant A + 3 VIX-family)

In [ ]:
import sys, json, subprocess
from pathlib import Path
sys.path.insert(0, str(Path(REPO_DIR)))
import config

VARIANTS = [
    {'tag': 'As', 'features': config.LSTM_VARIANT_A_FEATURES,
     'rp_path': 'data/processed/regime_probabilities_As.parquet',
     'meta_path': 'models/hmm_meta_As.joblib'},
    {'tag': 'Bs', 'features': config.LSTM_VARIANT_B_FEATURES,
     'rp_path': 'data/processed/regime_probabilities_Bs.parquet',
     'meta_path': 'models/hmm_meta_Bs.joblib'},
]
SEEDS  = [42, 43, 44]
SCRIPT = Path(REPO_DIR) / 'src' / 'train_LSTM_regime.py'

def _run(args_list, label):
    cmd = [sys.executable, '-u', str(SCRIPT), *args_list]
    print('=' * 80); print(f'[{label}]'); print('Command:', ' '.join(cmd)); print('-' * 80)
    proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end=''); lines.append(line)
    return proc.wait(), ''.join(lines)

for v in VARIANTS:
    tag, FEATS, RP, META = v['tag'], v['features'], v['rp_path'], v['meta_path']
    HPARAMS = Path(REPO_DIR) / 'models' / f'hparams_lstm_regime_{tag}.json'
    print(f'\n\n##############  Training variant {tag}  ##############')
    print(f'  features ({len(FEATS)}): {FEATS}')
    print(f'  output suffix: _{tag}_seedN, JSON: {HPARAMS.name}')
    for seed_idx, seed in enumerate(SEEDS):
        suffix = f'_{tag}_seed{seed}'
        args = ['--features', *FEATS, '--regime', 'both',
                '--output-suffix', suffix, '--seed', str(seed),
                '--regime-probs-path', RP, '--hmm-meta-path', META]
        if seed_idx > 0 and HPARAMS.exists():
            args += ['--fixed-hparams', str(HPARAMS)]
        rc, text = _run(args, f'variant {tag}, seed {seed}')
        if rc != 0:
            raise RuntimeError(f'variant {tag} seed {seed} failed (rc={rc})')
        if seed_idx == 0:
            marker = '=== Regime-Specific LSTM Results ==='
            idx = text.find(marker)
            if idx == -1:
                raise RuntimeError(f'{tag} seed 42: marker not found in stdout')
            HPARAMS.write_text(json.dumps(json.loads(text[idx + len(marker):].strip()), indent=2))
            print(f'\n[{tag} seed 42] hparams JSON saved → {HPARAMS.name}')

print('\n##############  All 12 regime LSTMs trained.  ##############')

## Cell 7 — Compute ensembles + within-variant DM (inline, both variants)

In [ ]:
import joblib, numpy as np, pandas as pd, torch, scipy.stats as sp_stats
from pathlib import Path
from src.lstm_model import LSTMRegressor, VolatilityWindowDataset
from torch.utils.data import DataLoader
from src.utils import regression_metrics
import config

TARGET   = config.LSTM_TARGET
test_df  = pd.read_parquet('data/processed/test.parquet')

def load_pt(suffix, regime):
    ckpt = torch.load(f'models/lstm_{regime}{suffix}.pt', map_location='cpu', weights_only=False)
    sc = joblib.load(f'models/lstm_{regime}{suffix}_scaler.joblib')
    return ckpt, sc

def _predict(ckpt, scaler, features):
    hp = ckpt['hyperparameters']
    sub = test_df[features + [TARGET]].dropna()
    X = scaler.transform(sub[features].to_numpy(np.float64))
    y = np.log(sub[TARGET].to_numpy(np.float64))
    ds = VolatilityWindowDataset(X, y, hp['seq_len'])
    ldr = DataLoader(ds, batch_size=hp['batch_size'], shuffle=False)
    model = LSTMRegressor(input_size=len(features), hidden_size=hp['hidden_size'],
                          n_layers=hp['n_layers'], dropout=hp['dropout'])
    model.load_state_dict(ckpt['state_dict']); model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in ldr: preds.append(model(x).numpy())
    pred = np.exp(np.concatenate(preds))
    return pd.Series(pred, index=sub.index[hp['seq_len'] - 1:])

def diebold_mariano(y_true, y_a, y_b, h=21, loss='mse'):
    y_true, y_a, y_b = (np.asarray(x, float) for x in (y_true, y_a, y_b))
    e_a = (y_a - y_true)**2 if loss == 'mse' else np.abs(y_a - y_true)
    e_b = (y_b - y_true)**2 if loss == 'mse' else np.abs(y_b - y_true)
    d = e_a - e_b; n = len(d); d_bar = float(d.mean())
    L = max(h - 1, 0); S = float(np.var(d, ddof=0))
    for k in range(1, L + 1):
        w = 1.0 - k / (L + 1)
        S += 2.0 * w * float(np.mean((d[k:] - d_bar) * (d[:-k] - d_bar)))
    if S <= 0: S = float(np.var(d, ddof=0))
    dm_raw = d_bar / np.sqrt(S / n)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return float(dm_raw * hln), float(2.0 * (1.0 - sp_stats.t.cdf(np.abs(dm_raw * hln), df=n - 1)))

VARIANT_CFG = {
    'As': {'features': config.LSTM_VARIANT_A_FEATURES,
           'baseline_suffix_per_seed': lambda s: f'_seed{s}',           # variant A baseline
           'rp_path': 'data/processed/regime_probabilities_As.parquet'},
    'Bs': {'features': config.LSTM_VARIANT_B_FEATURES,
           'baseline_suffix_per_seed': lambda s: f'_B_seed{s}',         # variant B baseline
           'rp_path': 'data/processed/regime_probabilities_Bs.parquet'},
}

for tag, cfg in VARIANT_CFG.items():
    print(f'\n=== Variant {tag} ensemble + DM ===')
    rp = pd.read_parquet(cfg['rp_path'])
    seed_dfs = {}
    for seed in (42, 43, 44):
        bsuf = cfg['baseline_suffix_per_seed'](seed)
        rsuf = f'_{tag}_seed{seed}'
        base_ck, base_sc = load_pt(bsuf, 'baseline')
        calm_ck, calm_sc = load_pt(rsuf, 'calm')
        vol_ck,  vol_sc  = load_pt(rsuf, 'volatile')
        pb = _predict(base_ck, base_sc, cfg['features'])
        pc = _predict(calm_ck, calm_sc, cfg['features'])
        pv = _predict(vol_ck,  vol_sc,  cfg['features'])
        df = pd.concat([pb.rename('baseline'), pc.rename('calm'), pv.rename('volatile')], axis=1)
        df['target'] = test_df[TARGET]
        df = df.join(rp[['p_calm', 'p_volatile']], how='inner').dropna()
        df['ensemble'] = df['p_calm']*df['calm'] + df['p_volatile']*df['volatile']
        out = Path(f'data/processed/seeds/test_predictions_{tag}_seed{seed}.parquet')
        out.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(out)
        print(f'  Wrote {out}  ({len(df)} rows)')
        seed_dfs[seed] = df

    common_idx = seed_dfs[42].index
    for s in (43, 44): common_idx = common_idx.intersection(seed_dfs[s].index)
    mean_df = pd.DataFrame(index=common_idx)
    mean_df['target']   = seed_dfs[42].loc[common_idx, 'target']
    for col in ['baseline', 'calm', 'volatile', 'ensemble']:
        mean_df[col] = sum(seed_dfs[s].loc[common_idx, col] for s in (42,43,44)) / 3
    mean_df = mean_df.join(rp[['p_calm', 'p_volatile']], how='inner').dropna()
    mean_path = Path(f'data/processed/test_predictions_{tag}.parquet')
    mean_df.to_parquet(mean_path)
    print(f'\n  Wrote mean-of-3-seeds: {mean_path}  ({len(mean_df)} rows)')

    y, bl, en = mean_df['target'].values, mean_df['baseline'].values, mean_df['ensemble'].values
    dm_mse, p_mse = diebold_mariano(y, bl, en, h=21, loss='mse')
    dm_mae, p_mae = diebold_mariano(y, bl, en, h=21, loss='mae')
    print(f'  DM(baseline, ensemble)  MSE: DM={dm_mse:+.3f} p={p_mse:.4f}    '
          f'MAE: DM={dm_mae:+.3f} p={p_mae:.4f}')
    print(f'  Mean-of-seeds metrics: ' +
          ', '.join(f'{k}={v:.4f}' for k, v in regression_metrics(y, en).items()))

print('\nAll variant As/Bs ensembles + DM tests complete.')

## Cell 8 — Sync new variant As/Bs artifacts back to Drive

In [ ]:
import shutil, glob
from pathlib import Path

drive_models = Path(PROJECT_DRIVE) / 'models'
drive_proc   = Path(PROJECT_DRIVE) / 'data' / 'processed'
drive_seeds  = drive_proc / 'seeds'
for p in (drive_models, drive_proc, drive_seeds):
    p.mkdir(parents=True, exist_ok=True)

patterns = [
    ('models', 'lstm_calm_As_seed*.pt'),
    ('models', 'lstm_calm_As_seed*_scaler.joblib'),
    ('models', 'lstm_volatile_As_seed*.pt'),
    ('models', 'lstm_volatile_As_seed*_scaler.joblib'),
    ('models', 'lstm_calm_Bs_seed*.pt'),
    ('models', 'lstm_calm_Bs_seed*_scaler.joblib'),
    ('models', 'lstm_volatile_Bs_seed*.pt'),
    ('models', 'lstm_volatile_Bs_seed*_scaler.joblib'),
    ('models', 'hmm_*_As.joblib'),
    ('models', 'hmm_*_Bs.joblib'),
    ('models', 'hparams_lstm_regime_As.json'),
    ('models', 'hparams_lstm_regime_Bs.json'),
    ('data/processed', 'regime_probabilities_As.parquet'),
    ('data/processed', 'regime_probabilities_Bs.parquet'),
    ('data/processed', 'test_predictions_As.parquet'),
    ('data/processed', 'test_predictions_Bs.parquet'),
    ('data/processed/seeds', 'test_predictions_As_seed*.parquet'),
    ('data/processed/seeds', 'test_predictions_Bs_seed*.parquet'),
]
for src_root, pattern in patterns:
    for f in glob.glob(f'{src_root}/{pattern}'):
        dst = (drive_models if src_root == 'models'
               else drive_seeds if src_root.endswith('seeds')
               else drive_proc) / Path(f).name
        shutil.copy2(f, dst)
        print(f'  → {dst}')
print('\nDone. Pull these into your local repo to regenerate paper figures and tables.')